In [1]:
# Importing libraries

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
# Load Dataset - Load the cleaned ETA dataset into a pandas DataFrame which will be used throughout the model training process.

df = pd.read_csv("ETA_Project_10k_Clean.csv")

print(df.shape)
df.head()

(10000, 16)


,trip_id,pickup_location,drop_location,pickup_date,pickup_time,pickup_hour,time_of_day,weekday,is_weekend,season,passenger_count,trip_distance_km,traffic_level,actual_eta_minutes,average_speed,surge_multiplier
0,id2875421,Bedford-Stuyvesant,Harlem,2016-03-14,17:24,17,2. Afternoon,Monday,0,2. Spring,1,1.96,High,7.6,15.5,1.8
1,id2377394,Chelsea,South Slope,2016-06-12,00:43,0,4. Night,Sunday,1,3. Summer,1,2.27,Low,11.0,12.4,1.0
2,id3858529,Upper West Side,East Harlem,2016-01-19,11:35,11,1. Morning,Tuesday,0,1. Winter,1,6.65,Low,35.4,11.3,1.0
3,id3504673,East Harlem,Bedford-Stuyvesant,2016-04-06,19:32,19,3. Evening,Wednesday,0,2. Spring,1,1.49,High,7.2,12.4,1.8
4,id2181028,South Slope,East Harlem,2016-03-26,13:30,13,2. Afternoon,Saturday,1,2. Spring,1,1.19,Low,7.2,9.9,1.0


In [3]:
# Basic Data Validation - Check for missing values and duplicate records to ensure the dataset is clean before preprocessing.
print("Missing Values")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Missing Values
trip_id               0
pickup_location       0
drop_location         0
pickup_date           0
pickup_time           0
pickup_hour           0
time_of_day           0
weekday               0
is_weekend            0
season                0
passenger_count       0
trip_distance_km      0
traffic_level         0
actual_eta_minutes    0
average_speed         0
surge_multiplier      0
dtype: int64

Duplicate Rows: 0


In [4]:
# Feature Cleaning -Clean categorical values by removing numeric prefixes and drop the unique Trip ID since it has no predictive value.
df["time_of_day"] = df["time_of_day"].str.replace(
    r"^\d+\.\s*", "", regex=True
)

df["season"] = df["season"].str.replace(
    r"^\d+\.\s*", "", regex=True
)

df.drop(columns=["trip_id"], inplace=True)

In [5]:
# Feature Engineering - Extract useful information from the pickup date and pickup time, then remove the original columns to reduce redundancy.

df["pickup_date"] = pd.to_datetime(df["pickup_date"])

df["month"] = df["pickup_date"].dt.month
df["day"] = df["pickup_date"].dt.day
df["day_of_year"] = df["pickup_date"].dt.dayofyear

df.drop(columns=["pickup_date"], inplace=True)

df["pickup_time"] = pd.to_datetime(
    df["pickup_time"],
    format="%H:%M"
)

df["pickup_minute"] = df["pickup_time"].dt.minute

df.drop(columns=["pickup_time"], inplace=True)

In [6]:
# Data Cleaning & Feature Selection - Remove unrealistic ETA values and drop features that add little predictive value or introduce redundancy.
df = df[df["actual_eta_minutes"] < 200]

df.drop(columns=["average_speed"], inplace=True)
df.drop(columns=["time_of_day"], inplace=True)

In [7]:
# Handle Rare Categories - Group infrequent pickup and drop locations into an "Other" category to improve generalization during prediction.
location_threshold = 20

pickup_counts = df["pickup_location"].value_counts()
rare_pickups = pickup_counts[pickup_counts < location_threshold].index

df["pickup_location"] = df["pickup_location"].replace(
    rare_pickups,
    "Other"
)

drop_counts = df["drop_location"].value_counts()
rare_drops = drop_counts[drop_counts < location_threshold].index

df["drop_location"] = df["drop_location"].replace(
    rare_drops,
    "Other"
)

In [8]:
# Train Test Split

X = df.drop("actual_eta_minutes", axis=1)
y = df["actual_eta_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [9]:
# Encode Categorical Features - Convert categorical variables into numerical format using ordinal encoding for traffic level and One-Hot Encoding for the remaining categorical features.

traffic_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

X_train["traffic_level"] = X_train["traffic_level"].map(traffic_map)
X_test["traffic_level"] = X_test["traffic_level"].map(traffic_map)

categorical_columns = [
    "pickup_location",
    "drop_location",
    "weekday",
    "season"
]

encoder = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

encoder.fit(X_train[categorical_columns])

encoded_train = pd.DataFrame(
    encoder.transform(X_train[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_train.index
)

encoded_test = pd.DataFrame(
    encoder.transform(X_test[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_test.index
)

X_train = X_train.drop(columns=categorical_columns)
X_test = X_test.drop(columns=categorical_columns)

X_train = pd.concat([X_train, encoded_train], axis=1)
X_test = pd.concat([X_test, encoded_test], axis=1)

In [10]:
# Train Model 
# Train the Gradient Boosting Regressor, which was selected as the best-performing model during experimentation.

model = GradientBoostingRegressor(
    random_state=42
)

model.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [11]:
# Evaluate Model - Evaluate the trained model using R² Score and Mean Absolute Error on both training and testing datasets.

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train R² :", r2_score(y_train, train_pred))
print("Test R²  :", r2_score(y_test, test_pred))

print("Train MAE:", mean_absolute_error(y_train, train_pred))
print("Test MAE :", mean_absolute_error(y_test, test_pred))

Train R² : 0.7414665089735192
Test R²  : 0.7202449954644861
Train MAE: 3.7290484928546705
Test MAE : 3.86863152308882


In [12]:
# Save Model & Encoder - Save the trained model and encoder so they can be reused directly during prediction without retraining.
import pickle

pickle.dump(
    model,
    open("gradient_boosting_model.pkl", "wb")
)

pickle.dump(
    encoder,
    open("encoder.pkl", "wb")
)

print("Model Saved Successfully!")

Model Saved Successfully!
